In [2]:
import pandas as pd
import bottleneck as bn
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
import numpy.typing as npt
import glob
from pathlib import Path
from multiprocessing import Pool, cpu_count
from typing import List, Tuple, Dict
import re
import math
import scipy.ndimage as sn
import scipy.signal as ss
import matplotlib.pyplot as plt
import plotly.io as pio


In [3]:
def process_file(file: str) -> pd.DataFrame:
    file_path = Path(file)
    parts = file_path.name.split(".")
    chromosome = "".join(parts[1:3])  # e.g., ['chr10', '1'] → 'chr101'

    df = pd.read_table(file_path, sep="\t", header=None)
    df.insert(0, "CHROM", chromosome)
    df.rename(columns={0: "Start", 1: "End", 2: "Rho"}, inplace=True)
    return df


def read_recombination_maps(files: str) -> List[pd.DataFrame]:
    file_names = glob.glob(files, recursive=True)
    with Pool(processes=cpu_count()//2) as pool:
        dfs = pool.map(process_file, file_names)
    return dfs

recombination_rate = read_recombination_maps("../output/blackcap_recombination_maps/n100*")


In [4]:
def sort_key(df: pd.DataFrame) -> int | float:
    chrom = str(df.iloc[0, 0]).replace("chr", "")
    try:
        return int(chrom)
    except ValueError:
        return float("inf")

sorted_rec_maps: List[pd.DataFrame] = sorted(recombination_rate, key=sort_key)

In [19]:
def make_windows(data_frame: pd.DataFrame, w_size: int) -> pd.DataFrame:
    if w_size is None or w_size <= 0:
        raise ValueError("w_size must be a positive integer.")
    if data_frame is None or len(data_frame) == 0:
        return pd.DataFrame(columns=["chrom", "start", "end", "cM/Mb", "midpoint"]).astype({
            "chrom": str, "start": "Int64", "end": "Int64", "cM/Mb": float, "midpoint": "Int64"
        })

    raw = data_frame.to_numpy()
    # Ensure single chromosome assumption holds
    chroms = pd.Series(raw[:, 0]).astype(str).unique()
    if len(chroms) != 1:
        raise ValueError(f"Expected a single chromosome, found {list(chroms)}.")
    chrom = str(chroms[0])

    starts = raw[:, 1].astype(np.int64)
    ends = raw[:, 2].astype(np.int64)
    values = raw[:, 3].astype(float)

    midpoints = (starts + ends) // 2
    bin_ids = (midpoints // w_size).astype(np.int64)

    # Group by bin using np.unique + inverse indices
    unique_bins, inverse_indices = np.unique(bin_ids, return_inverse=True)

    binned_sums = np.zeros(len(unique_bins), dtype=float)
    binned_count = np.zeros(len(unique_bins), dtype=np.int64)

    np.add.at(binned_sums, inverse_indices, values)
    np.add.at(binned_count, inverse_indices, 1)

    # Mean per occupied bin; scaling kept as in your original code
    binned_means = (binned_sums / binned_count) * 100 * 1e6

    win_starts = unique_bins * w_size
    win_ends = win_starts + w_size
    win_midpoints = (win_starts + win_ends) // 2

    out = pd.DataFrame({
        "chrom": chrom,
        "start": win_starts,
        "end": win_ends,
        "cM/Mb": binned_means,
        "midpoint": win_midpoints,
    })
    return out.astype({
        "chrom": str,
        "start": np.int64,
        "end": np.int64,
        "cM/Mb": float,
        "midpoint": np.int64
    })

windows_5kb = [make_windows(chrom, 5000) for chrom in sorted_rec_maps]

In [16]:
def _compute_midpoint(DataFrame: pd.DataFrame) -> pd.DataFrame:
    DataFrame['Midpoint'] = (DataFrame['Start'] + DataFrame['End']) // 2
    return DataFrame

def interpolate_recombination_rate(
        sorted_recombination_maps: list[pd.DataFrame] | pd.DataFrame
        ) -> tuple[list[npt.NDArray[np.float64]], list[npt.NDArray[np.int64]]]:

    if isinstance(sorted_recombination_maps, pd.DataFrame):
        sorted_recombination_maps = [sorted_recombination_maps]
    
    recombination_maps: list[pd.DataFrame] = [_compute_midpoint(df) for df in sorted_recombination_maps]
    recombination_rates: list[npt.NDArray[np.float64]] = [df['Rho'].to_numpy(dtype=float) for df in recombination_maps]
    midpoints: list[npt.NDArray[np.int64]] = [df['Midpoint'].to_numpy(dtype=int) for df in recombination_maps]
    genomic_bins: list[npt.NDArray[np.int64]] = [np.arange(mid.min(), mid.max() +50, 50) for mid in midpoints]
    binned_rec_rates = [
        np.interp(bins, mid, rho)
        for bins, mid, rho in zip(genomic_bins, midpoints, recombination_rates)
    ] 
    binned_rec_rates = [rec_rate * 1e8 for rec_rate in binned_rec_rates]
    return binned_rec_rates, genomic_bins

interpolated_recombination_maps = interpolate_recombination_rate(sorted_rec_maps)

In [17]:
def _compute_optimal_threshold(
    data: npt.NDArray[np.float64],
    k_min: float,
    k_max: float,
    sigma: float) -> float:
    # Build k grid (ensure at least two points)
    k = np.arange(k_min, k_max, 1.0, dtype=float)
    if k.size < 2:
        # fall back to a single candidate
        k = np.array([k_min], dtype=float)

    n_peaks = np.empty(k.shape, dtype=np.int64)

    # Count peaks for each candidate
    for idx, ki in enumerate(k):
        # Tune wlen/distance as you like; ensure they are <= len(data) where applicable
        peaks, _ = ss.find_peaks(data, prominence=ki * sigma, wlen=50_001, distance=10_000)
        n_peaks[idx] = peaks.size

    # Normalize x in [0,1]; guard zero range
    denom_x = (k.max() - k.min())
    x = (k - k.min()) / denom_x if denom_x != 0 else np.zeros_like(k, dtype=float)

    # Normalize y in [0,1]; guard zero range
    denom_y = (n_peaks.max() - n_peaks.min())
    y = (n_peaks - n_peaks.min()) / denom_y if denom_y != 0 else np.zeros_like(n_peaks, dtype=float)

    # Line from first to last point
    x0, y0 = x[0], y[0]
    x1, y1 = x[-1], y[-1]

    # Perpendicular distance of each (x,y) to the first-last line (a standard "kneedle"/L-method style)
    num = np.abs((y1 - y0) * x - (x1 - x0) * y + x1 * y0 - y1 * x0)
    den = np.hypot(y1 - y0, x1 - x0)

    # If all points are identical, just pick the first k
    if den == 0:
        return float(k[0])

    dist = num / den
    idx_star = int(np.argmax(dist))
    return float(k[idx_star])

def _calculate_robust_sigma(
    smoothed_signal: npt.NDArray[np.float64], 
    window_size: int) -> float:
    
    if window_size % 2 == 0:
        raise ValueError("Error: window_size must be an odd number")
    baseline: npt.NDArray[np.float64] = sn.median_filter(smoothed_signal, size=window_size, mode='reflect')
    contrast: npt.NDArray[np.float64] = smoothed_signal - baseline
    mad = np.median(np.abs(contrast - np.median(contrast))) + 1e-12
    sigma = 1.4826 * mad
    return float(sigma)

def call_hotspots(raw_signal: npt.NDArray[np.float64]) -> tuple[npt.NDArray[np.float64], npt.NDArray[np.int64], dict[str, npt.NDArray[np.float64]]]:
    
    
    smoothed_signal: npt.NDArray[np.float64] = ss.savgol_filter(raw_signal, window_length=2001, polyorder=1)
    robust_sigma = float(_calculate_robust_sigma(smoothed_signal, window_size=50_001))
    optimal_threshold = float(_compute_optimal_threshold(smoothed_signal, k_min=1, k_max=30, sigma=robust_sigma))
    peaks, properties = ss.find_peaks(
        smoothed_signal, 
        prominence=optimal_threshold*robust_sigma, 
        wlen=50_001
        )
    peaks = peaks.astype(np.int64, copy=False)
    properties = {k: np.asarray(v, dtype=np.float64) for k, v in properties.items()}
    return smoothed_signal, peaks, properties

def plot_recombination_hotspots(smoothed_signal: npt.NDArray[np.float64],
                                midpoint_bins: npt.NDArray[np.int64], 
                                peak_indeces: list[int], 
                                chromosome_number: int | None = None,
                                output_name: str = 'recombination_hotspots.png'):
    
    plt.style.use('seaborn-v0_8-darkgrid')
    plt.figure(figsize=(12,6), dpi=200)
    
    plt.plot(midpoint_bins, smoothed_signal, color='black', lw=1, alpha=0.4)
    plt.scatter(midpoint_bins[peak_indeces], smoothed_signal[peak_indeces], marker='X', s=40, color='red', edgecolors='black')
    plt.xlabel("Genomic Position")
    plt.ylabel(r'Recombination Rate  $\frac{\mathrm{cM}}{\mathrm{Mb}}$')
    plt.title(f'Chromosome {chromosome_number}')
    plt.tight_layout()
    plt.savefig(output_name)

In [ ]:
def chr_sort_key(df):
    # Ensure we're getting a string from the chrom column
    chrom = str(df.iloc[0]["chrom"]) if "chrom" in df.columns else str(df.iloc[0, 0])
    match = re.match(r"chr(\d+)", chrom)
    if match:
        return int(match.group(1))  # Numeric chromosomes
    else:
        return float("inf")         # Push non-numeric chromosomes (e.g., chrX) to the end

In [ ]:
def save_transformed_data(datasets: List[pd.DataFrame]):
    for data in datasets:
        try:
            chrom_name = data["chrom"].iat[0]
            window_size = data["end"].iat[0] - data["start"].iat[0]
            filename = f"transformed_{chrom_name}_w{window_size}.tsv"
            data.to_csv(filename, sep="\t", header=False, index=False)
        except Exception as e:
            print(f"Failed to save: {data}")
            raise e

NameError: name 'windows_10kb' is not defined

In [ ]:
def plot_genome_wide_rho(windows: list[pd.DataFrame], plot_name: str | None = None) -> None:
    if isinstance(windows, pd.DataFrame):
        windows = [windows]
    n = len(windows)
    cols = 1
    rows = n
    
    chrom_lengths = [int(df['midpoint'].max()) - int(df['midpoint'].min()) for df in windows]
    max_len = max(chrom_lengths)
    
    
    figure = make_subplots(rows=rows, 
                      cols=cols, 
                      subplot_titles=[f"Chromosome {i}" for i in range(1, n+1)], 
                      shared_yaxes=True,
                      shared_xaxes=False)
    
    for i, df in enumerate(windows):
        tick_step = 10_000_000
        tickvals = list(range(0, max_len + tick_step, tick_step))
        ticktext = list(str(i // 10_000_000) for i in tickvals)
        row = i + 1
        
        figure.add_trace(
            go.Scatter(x = df["midpoint"].astype(int),
                       y = df.iloc[:, 3].astype(float),
                       name = f"Chromosome {i}",
                       mode='lines',
                       line = dict(width=1, color="#722f37")),
            row,
            col = 1
        )
        
        figure.update_xaxes(
            title_text = "Genomic Position (Mb)",
            title_font = dict(size = 14, color = "black"),
            range=[0 - tick_step, max_len + tick_step],
            tickvals=tickvals,
            ticktext=ticktext,
            tickfont = dict(size=15, color='black'),
            ticks='outside',
            showline=False,
            linecolor='black',
            linewidth=2,
            row=row,
            col=1,
            gridcolor='white',
            showgrid=False
        )
        
    figure.update_layout(
        height=300 * rows,
        width=1900 * cols,
        title = {
            'text':'Recombination Rate over Genomic Positions',
            'x': 0.5,
            'y': 0.999,
            'yanchor': 'top',
            'xanchor': 'center',
            'font': dict(size=26, color = "black")
            },
        showlegend=False,
        margin=dict(t=80, l=20, r=20, b=20),
        plot_bgcolor='#f5f5f5'
    )
       
    figure.update_yaxes(
        title_text="cM/Mb",
        ticks='outside',
        tickfont=dict(size=15, color='black'),
        title_font = dict(size = 14, color = "black"),
        showline=False,
        linecolor='black',
        linewidth=2,
        gridcolor='white',
        showgrid=False)
    
    if plot_name:
        figure.write_html(plot_name, auto_open=True)
    figure.show()
        

NameError: name 'List' is not defined

In [12]:
def plot_rho_distribution(data: List[pd.DataFrame], plot_name: str | None = None):
    n = len(data)
    cols = 4
    rows = math.ceil(n/cols)
    figure = make_subplots(rows=rows,
                           cols=cols,
                           subplot_titles=[df["chrom"].iat[0] for df in data],
                           shared_yaxes=True)
    
    for i, df in enumerate(data):
        row = i // cols + 1
        col = i % cols + 1
        
        figure.add_trace(
            go.Histogram(x=df["cM/Mb"].astype(float), name=f"{df['chrom'].iat[0]}", showlegend=False, histnorm="probability"),
            row,
            col
        )
        
    figure.update_layout(
        height=300 * rows,
        width=300 * cols,
        title_text="Recombination Rate (cM/Mb) Distribution by Chromosome",
        bargap=0.2
    )
    figure.update_xaxes(title_text="cM/Mb")
    figure.update_yaxes(title_text="Frequency")
    
    if plot_name:
        try:
            figure.write_image(plot_name)
        except Exception as e:
            raise RuntimeError(f"Failed to save plot to {plot_name}: {e}")
    
    figure.show()

plot_rho_distribution(windows_5kb)